# Optional PCA Analysis from Shared MLP-Ready MFCC Cache

This optional notebook no longer scans audio folders, creates its own split, or extracts MFCCs. It loads the MLP-ready `80-D` MFCC mean/std cache produced from the shared raw MFCC foundation and uses PCA for feature inspection only.


## 1. Imports and Paths


In [ ]:
# Purpose: Imports the lightweight tools needed for PCA inspection of the shared MLP-ready
# feature cache. No Librosa/audio loading is used in this notebook.
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
MLP_CACHE_DIR = PROJECT_ROOT / "outputs" / "mlp" / "cache"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared" / "pca_analysis"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    # Purpose: Stops the notebook early with a clear message when a required upstream artifact is missing.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the shared analysis notebook, then 00_MLP_Data_Preparation.ipynb."
        )
    return path


## 2. Load Shared MLP-Ready Features


In [ ]:
# Purpose: Loads the train/validation features that were derived from the canonical split and
# shared raw MFCC cache. The test cache is intentionally not loaded for exploratory PCA plots.
for required in [
    MLP_CACHE_DIR / "X_train.npy",
    MLP_CACHE_DIR / "y_train.npy",
    MLP_CACHE_DIR / "train_metadata.csv",
    MLP_CACHE_DIR / "X_validation.npy",
    MLP_CACHE_DIR / "y_validation.npy",
    MLP_CACHE_DIR / "validation_metadata.csv",
    MLP_CACHE_DIR / "feature_config.json",
]:
    require_file(required)

X_train = np.load(MLP_CACHE_DIR / "X_train.npy")
y_train = np.load(MLP_CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(MLP_CACHE_DIR / "train_metadata.csv")
X_validation = np.load(MLP_CACHE_DIR / "X_validation.npy")
y_validation = np.load(MLP_CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(MLP_CACHE_DIR / "validation_metadata.csv")
with open(MLP_CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)

print("Loaded MLP-ready features:", MLP_CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
display(pd.DataFrame({"split": ["train", "validation"], "rows": [len(y_train), len(y_validation)]}))


## 3. Fit PCA on Training Features Only


In [ ]:
# Purpose: Fits PCA only on the training feature vectors. Validation is transformed using the
# fitted PCA object for visual inspection, not for model selection or test scoring.
pca_full = PCA(random_state=RANDOM_STATE)
X_train_pca_full = pca_full.fit_transform(X_train)
explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)
n_components_95 = int(np.searchsorted(cumulative_variance, 0.95) + 1)

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca_2d = pca_2d.fit_transform(X_train)
X_validation_pca_2d = pca_2d.transform(X_validation)

explained_variance_df = pd.DataFrame({
    "component": np.arange(1, len(explained_variance_ratio) + 1),
    "explained_variance_ratio": explained_variance_ratio,
    "cumulative_explained_variance": cumulative_variance,
})
display(explained_variance_df.head(20))
print("Components needed for 95 percent training variance:", n_components_95)


## 4. PCA Plots


In [ ]:
# Purpose: Saves PCA plots for feature inspection. These plots use train and validation rows
# from the canonical split, not a model-local split.
plt.figure(figsize=(8, 5))
plt.plot(explained_variance_df["component"], explained_variance_df["cumulative_explained_variance"], marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95 percent variance")
plt.xlabel("PCA component")
plt.ylabel("Cumulative explained variance")
plt.title("PCA cumulative explained variance on MLP-ready MFCC features")
plt.legend()
plt.tight_layout()
# Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same outputs.
plt.savefig(FIGURES_DIR / "pca_cumulative_explained_variance.png", dpi=300, bbox_inches="tight")
plt.show()

validation_plot_df = pd.DataFrame({
    "pc1": X_validation_pca_2d[:, 0],
    "pc2": X_validation_pca_2d[:, 1],
    "label": y_validation,
})
validation_plot_df["class_name"] = validation_plot_df["label"].map(CLASS_NAMES)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=validation_plot_df, x="pc1", y="pc2", hue="class_name", alpha=0.7)
plt.title("Validation rows projected into first two PCA components")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pca_validation_scatter.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Save PCA Inspection Outputs


In [ ]:
# Purpose: Saves PCA inspection artifacts. No test labels or test metrics are produced here.
explained_variance_df.to_csv(TABLES_DIR / "pca_explained_variance.csv", index=False)
validation_plot_df.to_csv(TABLES_DIR / "pca_validation_projection.csv", index=False)
joblib.dump(pca_full, MODELS_DIR / "pca_full_train_only.joblib")
joblib.dump(pca_2d, MODELS_DIR / "pca_2d_train_only.joblib")

summary = {
    "source_cache": str(MLP_CACHE_DIR),
    "source_representation": feature_config.get("representation"),
    "train_rows": int(len(y_train)),
    "validation_rows": int(len(y_validation)),
    "feature_dim": int(X_train.shape[1]),
    "components_for_95_percent_training_variance": n_components_95,
    "test_set_used": False,
    "does_not_rescan_audio": True,
    "does_not_create_split": True,
    "does_not_extract_mfcc": True,
}
with open(TABLES_DIR / "pca_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

display(pd.DataFrame(list(summary.items()), columns=["key", "value"]))


## 6. Reproducibility Checks


In [ ]:
# Purpose: 6. Reproducibility Checks.
checks = {
    "loads_shared_mlp_ready_cache": str(MLP_CACHE_DIR),
    "canonical_split_inherited_from_shared_cache": True,
    "pca_fit_on_training_only": True,
    "validation_used_for_visual_inspection_only": True,
    "test_metrics_computed_here": False,
    "no_custom_helper_module_required": True,
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
